# 02. 저장된 raw 데이터 전처리
인터넷/API 호출 없이 반복 실행할 수 있습니다.

In [1]:
from pathlib import Path
import sys
import pandas as pd
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from src.preprocess import build_ml_dataset
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
final_df_raw = pd.read_parquet(RAW_DIR / 'final_df_raw.parquet')
collection_missing_df = pd.read_csv(RAW_DIR / 'final_missing_df.csv')
sp500_universe = pd.read_csv(RAW_DIR / 'sp500_universe.csv')
print(final_df_raw.shape, collection_missing_df.shape, sp500_universe.shape)

(1284440, 8) (1, 7) (503, 4)


In [3]:
final_df, coverage_df, final_missing_df = build_ml_dataset(
    final_df_raw, universe=sp500_universe, final_missing_df=collection_missing_df
)
display(final_df.head())
display(coverage_df.head())
display(final_missing_df.head())

,Date,Ticker,Open,High,Low,Close,Volume,source
0,2016-01-04,A,37.751921,37.871444,37.089928,37.411728,3287300,yahoo
1,2016-01-05,A,37.448514,37.650791,37.089936,37.283016,2587200,yahoo
2,2016-01-06,A,36.997993,37.687568,36.823298,37.448513,2103600,yahoo
3,2016-01-07,A,36.906036,36.915233,35.683192,35.857883,3504300,yahoo
4,2016-01-08,A,36.060178,36.510699,35.370603,35.480934,3736700,yahoo


,Ticker,n_rows,actual_start_date,actual_end_date,n_price_imputed,n_ohlc_inconsistent,n_volume_missing,sources,expected_rows_10y,coverage_10y,short_history,has_quality_issue,company,sector
0,A,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,Agilent Technologies,Health Care
1,AAPL,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,Apple Inc.,Information Technology
2,ABBV,2637,2016-01-04,2026-06-30,0,1,0,yahoo,2738,0.9631,False,True,AbbVie,Health Care
3,ABNB,1393,2020-12-10,2026-06-30,0,0,0,yahoo,2738,0.5088,True,False,Airbnb,Consumer Discretionary
4,ABT,2637,2016-01-04,2026-06-30,0,1,0,yahoo,2738,0.9631,False,True,Abbott Laboratories,Health Care


,Ticker,company,sector,fail_stage,fail_reason,n_rows_raw,n_rows_final
0,HONA,Honeywell Aerospace,Industrials,collection,yahoo: rows=11 | chart: None,0,0


In [4]:
final_df.to_parquet(PROCESSED_DIR / 'final_df.parquet', index=False)
coverage_df.to_csv(PROCESSED_DIR / 'coverage_df.csv', index=False)
final_missing_df.to_csv(PROCESSED_DIR / 'final_missing_df.csv', index=False)
print(f'processed 저장 완료: {PROCESSED_DIR}')

processed 저장 완료: C:\workspaces\lab_middle_project\data\processed


In [5]:
print(
    "수집 성공 종목:",
    final_df_raw["Ticker"].nunique(),
)
print(
    "수집 실패 종목:",
    collection_missing_df["ticker"].nunique(),
)
print(
    "전체 대상 종목:",
    sp500_universe["ticker"].nunique(),
)

수집 성공 종목: 502
수집 실패 종목: 1
전체 대상 종목: 503


# 8개 지표 통합 데이터프레임(60/120)

In [6]:
from src.feature import add_features

sp500_beta_df = pd.read_parquet(
    RAW_DIR / "sp500_beta_df.parquet"
)

In [7]:
final_60_df = add_features(
    prices=final_df,
    sp500_beta_df=sp500_beta_df,
    windows=(60,),
)

In [8]:
final_120_df = add_features(
    prices=final_df,
    sp500_beta_df=sp500_beta_df,
    windows=(120,),
)

In [9]:
final_60_df.to_parquet(
    PROCESSED_DIR / "final_60_df.parquet",
    index=False
)

final_120_df.to_parquet(
    PROCESSED_DIR / "final_120_df.parquet",
    index=False
)

In [10]:
final_120_df.head()

,Date,Ticker,Open,High,Low,Close,Volume,source,volatility_120d,mdd_120d,downside_volatility_120d,beta_120d,ma_gap_120d,rsi_120d,momentum_120d,return_120d,cagr_10y
0,2016-01-04,A,37.751921,37.871444,37.089928,37.411728,3287300,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2016-01-05,A,37.448514,37.650791,37.089936,37.283016,2587200,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2016-01-06,A,36.997993,37.687568,36.823298,37.448513,2103600,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2016-01-07,A,36.906036,36.915233,35.683192,35.857883,3504300,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2016-01-08,A,36.060178,36.510699,35.370603,35.480934,3736700,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [11]:
final_60_df.head()

,Date,Ticker,Open,High,Low,Close,Volume,source,volatility_60d,mdd_60d,downside_volatility_60d,beta_60d,ma_gap_60d,rsi_60d,momentum_60d,return_60d,cagr_10y
0,2016-01-04,A,37.751921,37.871444,37.089928,37.411728,3287300,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2016-01-05,A,37.448514,37.650791,37.089936,37.283016,2587200,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2016-01-06,A,36.997993,37.687568,36.823298,37.448513,2103600,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2016-01-07,A,36.906036,36.915233,35.683192,35.857883,3504300,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2016-01-08,A,36.060178,36.510699,35.370603,35.480934,3736700,yahoo,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [12]:
final_120_df.tail()

,Date,Ticker,Open,High,Low,Close,Volume,source,volatility_120d,mdd_120d,downside_volatility_120d,beta_120d,ma_gap_120d,rsi_120d,momentum_120d,return_120d,cagr_10y
1284435,2026-06-24,ZTS,77.271344,78.542482,76.824463,77.628853,4762100,yahoo,0.427708,-0.431405,0.388168,0.704060,-0.287251,38.757980,-0.376274,-0.376274,0.060513
1284436,2026-06-25,ZTS,78.165119,79.356816,76.725160,77.281281,5790300,yahoo,0.427707,-0.431405,0.388164,0.707814,-0.287903,38.761887,-0.376155,-0.376155,0.059970
1284437,2026-06-26,ZTS,76.357713,77.072729,75.026993,75.563248,15723300,yahoo,0.428515,-0.431405,0.389498,0.709400,-0.301136,38.284444,-0.390508,-0.390508,0.057500
1284438,2026-06-29,ZTS,76.298131,76.357713,72.554225,72.742912,6520300,yahoo,0.428825,-0.438836,0.393236,0.645953,-0.324376,36.656747,-0.428904,-0.428904,0.055330
1284439,2026-06-30,ZTS,72.663470,73.050770,70.975242,71.362541,8322200,yahoo,0.429310,-0.449485,0.394196,0.631164,-0.334311,36.300291,-0.439654,-0.439654,0.052250
